
# Step 01 — Semantic Attention Statistics (A–E)

This notebook performs the first inferential analysis on the cleaned participant-level semantic attention dataset.

Primary variables:

\[
G=[g_H,g_P,g_C,g_O]
\]

and

\[
\text{Semantic Focus}=1-g_O=g_H+g_P+g_C.
\]

This step does **not** yet perform the compositional ILR analysis. That is kept separate in Step 02.


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print("Current working directory:", Path.cwd().resolve())


## 1. Locate the cleaned master dataset

In [ ]:

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for base in candidates:
        candidate = base / "semantic_attention_analysis_outputs"
        if candidate.is_dir():
            return base

    # Also check child folders in case notebook is launched from a parent directory.
    for base in candidates[:3]:
        try:
            for child in base.iterdir():
                if child.is_dir() and (child / "semantic_attention_analysis_outputs").is_dir():
                    return child
        except Exception:
            pass

    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate 'semantic_attention_analysis_outputs'. "
        "Run this notebook from inside your SEMANTIC project, or set PROJECT_ROOT manually."
    )

DATA_DIR = PROJECT_ROOT / "semantic_attention_analysis_outputs"

# Prefer QC-pass balanced dataset.
candidate_files = [
    DATA_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES_QC_PASS.csv",
    DATA_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES.csv",
]

DATA_FILE = next((p for p in candidate_files if p.exists()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "Could not find the balanced master dataset. Expected one of:\n"
        + "\n".join(str(p) for p in candidate_files)
    )

STEP_DIR = Path.cwd().resolve()
OUTPUT_DIR = STEP_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_FILE:", DATA_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Load and validate the analysis data

In [ ]:

df = pd.read_csv(DATA_FILE)

required = [
    "participant_id", "condition", "image_id",
    "g_H", "g_P", "g_C", "g_O"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for c in ["g_H", "g_P", "g_C", "g_O"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# If QC flags exist, keep valid rows only.
if "row_qc_pass" in df.columns:
    before = len(df)
    df = df[df["row_qc_pass"].astype(bool)].copy()
    print(f"QC filter: {before} -> {len(df)} rows")

df["semantic_focus"] = 1.0 - df["g_O"]

print("Rows:", len(df))
print("Conditions:", sorted(df["condition"].dropna().unique()))
print("Images:", df["image_id"].nunique())
print("Participants:", df["participant_id"].nunique())

display(df[[
    "participant_id", "condition", "image_id",
    "g_H", "g_P", "g_C", "g_O", "semantic_focus"
]].head())


## 3. Dataset structure

In [ ]:

cell_counts = (
    df.groupby(["image_id", "condition"])["participant_id"]
      .nunique()
      .unstack("condition")
)

print("Participant counts per image × condition:")
display(cell_counts)

condition_rows = (
    df.groupby("condition")
      .agg(
          rows=("participant_id", "size"),
          unique_participants=("participant_id", "nunique"),
          images=("image_id", "nunique")
      )
      .reset_index()
)

display(condition_rows)

cell_counts.to_csv(OUTPUT_DIR / "participant_counts_analysis_dataset.csv")
condition_rows.to_csv(OUTPUT_DIR / "condition_sample_sizes.csv", index=False)


## 4. Descriptive statistics

In [ ]:

OUTCOMES = ["g_H", "g_P", "g_C", "g_O", "semantic_focus"]

descriptive = (
    df.groupby("condition")[OUTCOMES]
      .agg(["count", "mean", "std", "median", "min", "max"])
)

display(descriptive)
descriptive.to_csv(OUTPUT_DIR / "descriptive_statistics_by_condition.csv")


## 5. 95% confidence intervals for condition means

In [ ]:

from scipy import stats

rows = []

for cond, sub in df.groupby("condition"):
    for outcome in OUTCOMES:
        x = sub[outcome].dropna().to_numpy()
        n = len(x)
        mean = np.mean(x)
        sd = np.std(x, ddof=1)
        se = sd / np.sqrt(n)
        tcrit = stats.t.ppf(0.975, df=n-1)
        rows.append({
            "condition": cond,
            "outcome": outcome,
            "n": n,
            "mean": mean,
            "sd": sd,
            "se": se,
            "ci95_low": mean - tcrit * se,
            "ci95_high": mean + tcrit * se,
        })

ci_table = pd.DataFrame(rows)
display(ci_table)
ci_table.to_csv(OUTPUT_DIR / "condition_means_95CI.csv", index=False)



## 6. Mixed-effects models

For each outcome:

\[
y_{p,i} = \beta_0 + \beta_{\text{Condition}} + u_p + v_i + \epsilon
\]

where:

- participant is a grouping random effect;
- advertisement/image is included as a variance component.

Condition A is used as the reference level.


In [ ]:

import statsmodels.formula.api as smf

# Force consistent reference ordering.
df["condition"] = pd.Categorical(
    df["condition"],
    categories=["A", "B", "C", "D", "E"],
    ordered=False
)

mixed_models = {}
mixed_model_tables = []

for outcome in OUTCOMES:
    print("\n" + "=" * 90)
    print("OUTCOME:", outcome)
    print("=" * 90)

    formula = f"{outcome} ~ C(condition, Treatment(reference='A'))"

    try:
        model = smf.mixedlm(
            formula,
            data=df,
            groups=df["participant_id"],
            vc_formula={"image": "0 + C(image_id)"},
            re_formula="1"
        )

        result = model.fit(
            reml=False,
            method="lbfgs",
            maxiter=2000,
            disp=False
        )

        mixed_models[outcome] = result
        print(result.summary())

        params = result.params
        conf = result.conf_int()
        pvals = result.pvalues

        table = pd.DataFrame({
            "term": params.index,
            "estimate": params.values,
            "ci95_low": conf.iloc[:, 0].values,
            "ci95_high": conf.iloc[:, 1].values,
            "p_value": pvals.values,
        })
        table.insert(0, "outcome", outcome)
        mixed_model_tables.append(table)

    except Exception as e:
        print("MODEL FAILED:", type(e).__name__, str(e))

if mixed_model_tables:
    mixed_results = pd.concat(mixed_model_tables, ignore_index=True)
    display(mixed_results)
    mixed_results.to_csv(
        OUTPUT_DIR / "mixed_effects_model_coefficients.csv",
        index=False
    )


## 7. Pairwise A–E comparisons with BH/FDR correction

In [ ]:

from itertools import combinations
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

pairwise_rows = []

# This provides a transparent participant-row level sensitivity analysis.
# The mixed model above remains the main inferential model.
for outcome in OUTCOMES:
    for c1, c2 in combinations(["A", "B", "C", "D", "E"], 2):
        x = df.loc[df["condition"] == c1, outcome].dropna().to_numpy()
        y = df.loc[df["condition"] == c2, outcome].dropna().to_numpy()

        stat, p = mannwhitneyu(x, y, alternative="two-sided")

        # Rank-biserial correlation from U.
        rbc = (2 * stat) / (len(x) * len(y)) - 1

        pairwise_rows.append({
            "outcome": outcome,
            "condition_1": c1,
            "condition_2": c2,
            "n_1": len(x),
            "n_2": len(y),
            "U": stat,
            "p_raw": p,
            "rank_biserial": rbc,
            "mean_1": np.mean(x),
            "mean_2": np.mean(y),
            "median_1": np.median(x),
            "median_2": np.median(y),
        })

pairwise = pd.DataFrame(pairwise_rows)

# BH correction separately within each outcome family.
pairwise["q_BH"] = np.nan
pairwise["reject_FDR_0_05"] = False

for outcome in OUTCOMES:
    mask = pairwise["outcome"] == outcome
    reject, qvals, _, _ = multipletests(
        pairwise.loc[mask, "p_raw"],
        alpha=0.05,
        method="fdr_bh"
    )
    pairwise.loc[mask, "q_BH"] = qvals
    pairwise.loc[mask, "reject_FDR_0_05"] = reject

display(pairwise.sort_values(["outcome", "q_BH"]))
pairwise.to_csv(
    OUTPUT_DIR / "pairwise_MannWhitney_BH_FDR.csv",
    index=False
)


## 8. Image-level sensitivity analysis

In [ ]:

# Average participants within each image × condition cell.
# This prevents images with 21 participants from receiving slightly more weight
# than images with 19 participants.

image_condition_means = (
    df.groupby(["image_id", "condition"], observed=True)[OUTCOMES]
      .mean()
      .reset_index()
)

display(image_condition_means.head(20))
image_condition_means.to_csv(
    OUTPUT_DIR / "image_condition_means.csv",
    index=False
)

image_summary = (
    image_condition_means
    .groupby("condition", observed=True)[OUTCOMES]
    .agg(["count", "mean", "std", "median"])
)

display(image_summary)
image_summary.to_csv(
    OUTPUT_DIR / "image_level_condition_summary.csv"
)


## 9. Publication-ready compact results table

In [ ]:

pub_rows = []

for cond, sub in df.groupby("condition", observed=True):
    for outcome in OUTCOMES:
        x = sub[outcome].dropna()
        pub_rows.append({
            "condition": cond,
            "metric": outcome,
            "N": len(x),
            "Mean": x.mean(),
            "SD": x.std(ddof=1),
            "Median": x.median(),
        })

publication_table = pd.DataFrame(pub_rows)

publication_table["Mean_SD"] = publication_table.apply(
    lambda r: f"{r['Mean']:.4f} ± {r['SD']:.4f}",
    axis=1
)

compact = publication_table.pivot(
    index="condition",
    columns="metric",
    values="Mean_SD"
)

display(compact)
compact.to_csv(
    OUTPUT_DIR / "publication_descriptive_table.csv"
)


## 10. Final QC / output summary

In [ ]:

print("STEP 01 COMPLETE")
print("=" * 70)
print("Analysis rows:", len(df))
print("Images:", df["image_id"].nunique())
print("Conditions:", list(df["condition"].dropna().unique()))
print("\nGenerated outputs:")

for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p.name)

print("\nNext step:")
print("02_compositional_analysis")
print("We will analyze G=[g_H,g_P,g_C,g_O] correctly as compositional data using log-ratio / ILR methods.")
